In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    f1_score,
    hamming_loss,
    jaccard_score,
)
from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/processed/feature_engineering_sample.csv")
MODELS_DIR = Path("../models")
OUTPUTS_DIR = Path("../outputs")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

label_cols = [
    "label_respiratory_risk",
    "label_cardiovascular_risk",
    "label_vulnerable_alert",
    "label_outdoor_warning",
    "label_industrial_event",
]

df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

df.head()

In [ ]:
target_df = df[label_cols].astype(int)

drop_cols = ["reading_id", "timestamp"] + label_cols
feature_df = df.drop(columns=drop_cols)

print("Feature shape:", feature_df.shape)
print("Target shape:", target_df.shape)
feature_df.head()

In [ ]:
df_sorted = df.sort_values("timestamp").reset_index(drop=True)

split_idx = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_idx].copy()
test_df = df_sorted.iloc[split_idx:].copy()

X_train = train_df.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)
y_train = train_df[label_cols].astype(int)
y_test = test_df[label_cols].astype(int)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

In [ ]:
categorical_features = ["season", "station_id", "station_type"]
numeric_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

In [ ]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", MultiOutputClassifier(
        RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
        n_jobs=-1,
    )),
])

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

baseline_proba = np.column_stack([
    estimator.predict_proba(
        baseline_model.named_steps["preprocessor"].transform(X_test)
    )[:, 1]
    for estimator in baseline_model.named_steps["model"].estimators_
])

In [ ]:
chain_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", ClassifierChain(
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42,
        ),
        order=[2, 0, 1, 3, 4],
        random_state=42,
    )),
])

chain_model.fit(X_train, y_train)
chain_pred = chain_model.predict(X_test)

chain_proba = chain_model.named_steps["model"].predict_proba(
    chain_model.named_steps["preprocessor"].transform(X_test)
)
chain_proba = np.column_stack(chain_proba)

In [ ]:
def evaluate_multilabel(y_true, y_pred, label_names):
    return {
        "hamming_loss": hamming_loss(y_true, y_pred),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "exact_match": (y_true.values == y_pred).all(axis=1).mean(),
        "jaccard_samples": jaccard_score(y_true, y_pred, average="samples", zero_division=0),
    }

label_names = [c.replace("label_", "") for c in label_cols]

results = pd.DataFrame([
    {"model": "Binary Relevance RF", **evaluate_multilabel(y_test, baseline_pred, label_names)},
    {"model": "Classifier Chain LR", **evaluate_multilabel(y_test, chain_pred, label_names)},
])

results

In [ ]:
results_plot = results.set_index("model")[["hamming_loss", "f1_micro", "f1_macro", "exact_match", "jaccard_samples"]]

ax = results_plot.plot(kind="bar", figsize=(12, 5), edgecolor="white")
ax.set_title("Multilabel Model Comparison")
ax.set_ylabel("Score")
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "model_comparison_metrics.png", dpi=300, bbox_inches="tight")

In [ ]:
baseline_per_label = f1_score(y_test, baseline_pred, average=None, zero_division=0)
chain_per_label = f1_score(y_test, chain_pred, average=None, zero_division=0)

per_label_df = pd.DataFrame({
    "label": label_names,
    "Binary Relevance RF": baseline_per_label,
    "Classifier Chain LR": chain_per_label,
}).set_index("label")

per_label_df

In [ ]:
ax = per_label_df.plot(kind="bar", figsize=(12, 5), edgecolor="white")
ax.set_title("Per-Label F1 Score by Model")
ax.set_ylabel("F1 Score")
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "per_label_f1_comparison.png", dpi=300, bbox_inches="tight")

In [ ]:
threshold_grid = np.arange(0.1, 0.9, 0.05)
best_thresholds = {}

for idx, label in enumerate(label_cols):
    best_score = -1
    best_threshold = 0.5
    for threshold in threshold_grid:
        preds = (baseline_proba[:, idx] >= threshold).astype(int)
        score = f1_score(y_test.iloc[:, idx], preds, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = threshold
    best_thresholds[label] = float(best_threshold)

threshold_df = pd.Series(best_thresholds, name="best_threshold").to_frame()
threshold_df

In [ ]:
tuned_pred = np.column_stack([
    (baseline_proba[:, idx] >= best_thresholds[label]).astype(int)
    for idx, label in enumerate(label_cols)
])

tuned_results = evaluate_multilabel(y_test, tuned_pred, label_names)
tuned_results

In [ ]:
summary = {
    "selected_model": "Binary Relevance RF",
    "metrics": {
        "baseline": evaluate_multilabel(y_test, baseline_pred, label_names),
        "chain": evaluate_multilabel(y_test, chain_pred, label_names),
        "tuned_baseline": tuned_results,
    },
    "thresholds": best_thresholds,
    "label_order": label_cols,
}

with open(MODELS_DIR / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with open(MODELS_DIR / "label_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(best_thresholds, f, indent=2)

print("Saved metrics_summary.json and label_thresholds.json")